# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SajidurCodes/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"C:\Projects AI ML\flyrank-ml-starter")

load_dotenv(PROJECT_ROOT / ".env", override=True)

HF_TOKEN = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The five contract answers, in plain words:**

1. **One row (unit of analysis).** For the Refresh / Content Opportunity Scoring lane, one row in my feature frame is **one content item (page) for one client, as of the decision month**. The raw warehouse grain is finer than that: `fact_content_daily_performance` is client × content × day. I aggregate that daily grain up to client × content × month before I rank anything — a page doesn't get "refreshed" daily, so daily rows aren't the decision unit, they're the raw material for it.

2. **Table(s).** Primary: `fact_content_daily_performance` (the daily fact table, partitioned by `month=YYYY-MM`) from `FlyRank/internship-warehouse`. I also touch the client dimension (`dim_clients`, for `gsc_data_start` / `ga4_data_start` — needed for the data-limits section) and, if I use query-level signals, `fact_content_query_90d`.

3. **Time window.** I develop on **`month=2026-03`**, a mid-panel month — not the `_sample` table and not the final month (`2026-06`), which the dataset card explicitly flags as the sealed outcome window and off-limits for label logic. All exploration, grain checks, and features below are computed on this one month unless noted.

4. **What I'd predict or rank (label / proxy).** The end goal is a **ranking of pages by review priority** — which pages most deserve a human look this month for refresh, expansion, protection, pruning, or monitoring. In this notebook I'm not training a model yet (that's a later week); the "label or proxy" I'm setting up for is something like *"this page's organic performance declined materially between the current month and a later month"* — a forward-looking, past→future comparison. I deliberately do **not** compute that comparison here using the same month I'm building features from — that would be the leakage trap in Part 4.

5. **One thing I deliberately exclude.** I exclude **any GA4-derived engagement/conversion metric for clients whose `ga4_data_start` is later than `2026-03`** (i.e., clients who don't have GA4 history for this month). Why: including it would silently turn "no GA4 signal" into "zero engagement," which isn't a fact about the page — it's a fact about onboarding order. Mixing that into ranking pages would bias the queue toward clients who joined GA4 early, not toward pages that actually need review. I check for this explicitly in Section 3.

In [12]:
import duckdb

con = duckdb.connect()
con.execute("CREATE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"

# 1a. What columns actually exist? Don't guess — ask.
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 0").df()
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I don't have every exact column name confirmed yet (fill in / correct against the `DESCRIBE` output above once you run this), but the bucket logic is set:

**Feature** (known at the decision moment, goes into the model later):
- Rolling/aggregated GSC signals from the *current* and *prior* months only — clicks, impressions, CTR, average position — never from a month after the decision point.
- Content metadata that doesn't change with performance: word count, publish date, days-since-last-updated.
- Client-level context that's stable within the month: industry/vertical, GSC/GA4 onboarding dates (used to *gate* availability, see below).

**Label / proxy** (what I'd eventually rank on — deliberately *not* built into the feature frame in this notebook):
- A forward-looking change in organic performance (e.g., clicks or position movement) measured between the decision month and a *later* month. This only exists once the later month has happened, so it can never be a feature.

**Context** (useful for interpretation / grouping, not fed to a model as a raw feature):
- Client identifier (namespaced/salted hash) — used to group and to hold out clients later, never as a numeric feature.
- Content identifier / URL hash — same role, an identifier not a signal.
- `month` partition value itself.

**Excluded** (deliberately left out, with why):
- GA4 engagement metrics for client-months before that client's `ga4_data_start` — see contract answer 5 above; this is missing-not-random, not "zero," and including it would confound the ranking.
- Anything from `fact_content_query_90d`'s trailing 90-day window that overlaps into the sealed final month — the 90-day window can quietly reach past the decision point depending on which month I anchor it to. I'm not using the query table in this notebook to avoid that trap entirely.
- The `_sample` table — it's exactly June 2026, the natural outcome window for any past→future label, per the dataset card's own warning. Mechanics-testing only, never label logic.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

sample = con.sql(f"SELECT * FROM read_parquet('{FACT}') LIMIT 20").df()
sample.head(20)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,True,False,True,<NA>,239,1,1756,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,True,False,True,<NA>,191,0,1496,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,True,False,True,<NA>,55,0,180,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,True,False,True,<NA>,77,0,434,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,True,False,True,<NA>,2,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries, each proving one claim from Section 1:

1. **Grain** — one row really is client × content × day for `month=2026-03` (no duplicate keys).
2. **Row count & date span** — how many rows are in this slice, and does the date range match a full March 2026.
3. **Availability** — filtering with `IS TRUE` on the GA4-availability flag, how many rows survive vs. how many get excluded (proving the exclusion from Section 2 is real and sized, not theoretical).

In [14]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                       AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_keys
    FROM read_parquet('{FACT}')
""").df()
print(grain_check)



span_check = con.sql(f"""
    SELECT
        COUNT(*)              AS row_count,
        MIN(report_date)      AS min_date,
        MAX(report_date)      AS max_date,
        COUNT(DISTINCT report_date) AS distinct_days
    FROM read_parquet('{FACT}')
""").df()
print(span_check)



availability_check = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS rows_with_ga4,
        COUNT(*)                                        AS rows_total,
        ROUND(100.0 * COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{FACT}')
""").df()
print(availability_check)

   total_rows  distinct_keys
0     9841378        9841378
   row_count   min_date   max_date  distinct_days
0    9841378 2026-03-01 2026-03-31             31
   rows_with_ga4  rows_total  pct_available
0        6822637     9841378           69.3


## 3b. Five features (max) + the leakage trap

*Build a small feature frame for this lane from `month=2026-03`. One line per feature: "knowable at the decision moment because…". Then: add one label-derived column on purpose, watch a quick score jump toward perfect, delete it, keep the honest number.*

**Five features, each with its "available when" line:**

1. `clicks_month` (sum of daily clicks) — *knowable at the decision moment because* it's a straight aggregation of GSC data already logged for the current month; nothing about it depends on what happens after decision day.
2. `avg_position_month` (mean daily average position) — *knowable because* GSC reports position daily as search results are served; it reflects only what already happened, not a future outcome.
3. `ctr_month` (clicks / impressions for the month) — *knowable because* both numerator and denominator are fully observed within the current month's logged data.
4. `days_since_last_update` (decision date − content's last-modified date) — *knowable because* it's a property of the content item's edit history up to today, not a future event.
5. `word_count` — *knowable because* it's a static content-metadata attribute measured at any point, including today.

Deliberately **not** a feature: anything computed from a month later than the decision month.

**The trap:** I add one label-derived column — e.g. `next_month_clicks_change` (change in clicks between March and a later month) — as if it were a feature, retrain a trivial quick score (a simple rule or correlation, not a full model — that's a later week), watch it jump toward a near-perfect fit, then remove the column and report the honest, much-lower number.

In [15]:
FEATURES = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)                                       AS clicks_month,
        AVG(gsc_avg_position)                                  AS avg_position_month,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)      AS ctr_month
        -- TODO: join in days_since_last_update and word_count from the content
        -- dimension table once you've confirmed its name/columns via DESCRIBE.
    FROM read_parquet('{FACT}')
    GROUP BY client_hash_id, content_hash_id
""").df()

FEATURES.head()

,client_hash_id,content_hash_id,clicks_month,avg_position_month,ctr_month
0,client_62f4a7e64f5e0096,content_764aeb5b0fa0d24e,1.0,6.925926,0.076923
1,client_62f4a7e64f5e0096,content_6fa4a9b6630f353f,0.0,15.558428,0.000000
2,client_62f4a7e64f5e0096,content_5081378458afb2ae,0.0,21.377778,0.000000
3,client_62f4a7e64f5e0096,content_8f5de64943db65bf,0.0,34.950186,0.000000
4,client_62f4a7e64f5e0096,content_c6de1f27c3bdc842,0.0,6.368345,0.000000


In [16]:
LATER_MONTH_FACT = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet"

later = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_next_month
    FROM read_parquet('{LATER_MONTH_FACT}')
    GROUP BY client_hash_id, content_hash_id
""").df()

leaky = FEATURES.merge(later, on=["client_hash_id", "content_hash_id"], how="inner")
leaky["next_month_clicks_change"] = leaky["clicks_next_month"] - leaky["clicks_month"]

import numpy as np
from scipy.stats import spearmanr

honest_corr, _ = spearmanr(leaky["clicks_month"], leaky["clicks_next_month"])
leaked_corr, _ = spearmanr(leaky["next_month_clicks_change"], leaky["clicks_next_month"])

print("Honest quick-score correlation (no leak):", round(honest_corr, 3))
print("Leaked quick-score correlation (with next_month_clicks_change as a 'feature'):", round(leaked_corr, 3))

FEATURES_HONEST = leaky.drop(columns=["next_month_clicks_change", "clicks_next_month"])
print("\nKept honest feature frame:")
FEATURES_HONEST.head()


Honest quick-score correlation (no leak): 0.72
Leaked quick-score correlation (with next_month_clicks_change as a 'feature'): 0.123

Kept honest feature frame:


,client_hash_id,content_hash_id,clicks_month,avg_position_month,ctr_month
0,client_62f4a7e64f5e0096,content_764aeb5b0fa0d24e,1.0,6.925926,0.076923
1,client_62f4a7e64f5e0096,content_6fa4a9b6630f353f,0.0,15.558428,0.000000
2,client_62f4a7e64f5e0096,content_5081378458afb2ae,0.0,21.377778,0.000000
3,client_62f4a7e64f5e0096,content_8f5de64943db65bf,0.0,34.950186,0.000000
4,client_62f4a7e64f5e0096,content_c6de1f27c3bdc842,0.0,6.368345,0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this is an **unbalanced panel** — per the dataset card, `dim_clients.gsc_data_start` / `ga4_data_start` differ by client, so "March 2026" does not mean the same thing for every client in the slice. A client whose GA4 history starts in April 2026 will show up in this month's rows with GSC-only data, not zero-engagement data. Any ranking that doesn't account for this will systematically penalize newer clients regardless of how their pages are actually performing.

Other limits this data can never resolve on its own:
- **Correlation, not causation.** A page's clicks moving after a "refresh" doesn't prove the refresh caused it — seasonality, algorithm updates, and competitor changes move at the same time.
- **No qualitative content signal.** The warehouse has performance numbers and light metadata, not the actual page content, so anything about *why* a page underperforms (bad structure, wrong intent match, thin content) has to be inferred, not read directly.
- **The final month (`2026-06`) is sealed.** Any label built past→future can only be validated up to the last available "later" month in the mid-panel window — I can't peek at June to check my logic without contaminating the eventual sealed test.

In [20]:
onboarding_gap = con.sql("""
    SELECT
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE ga4_data_start > DATE '2026-03-01') AS clients_missing_ga4_in_march
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""")
print(onboarding_gap)

┌───────────────┬──────────────────────────────┐
│ total_clients │ clients_missing_ga4_in_march │
│     int64     │            int64             │
├───────────────┼──────────────────────────────┤
│           104 │                           25 │
└───────────────┴──────────────────────────────┘



In [21]:
content_schema = con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') LIMIT 0
""")
print(content_schema)

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.